# YOLO26 ThermalUAV2UAV baseline (Colab)

1. Set `REPO_URL` to your GitHub repo.
2. Runtime → Change runtime type → GPU.
3. Run all cells.
4. Download `reports/<run>/` and `reports/compare/` (optionally sync `runs/` to Drive).

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/Ther-Inf-multiclass-detection.git"
BRANCH = "main"
EPOCHS = 20
IMGSZ = 640
BATCH = 16
QUANTIZE = "fp16"  # fp32 | fp16 | int8 (TensorRT latency export; training stays float)
RUN_NAME = "uav2uav_yolo26n_baseline"
SAVE_RUNS_TO_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/ther_inf_runs"

In [ ]:
from google.colab import drive
import os

if SAVE_RUNS_TO_DRIVE:
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
import os
import pathlib

os.chdir("/content")
!rm -rf repo
!git clone --branch {BRANCH} {REPO_URL} repo

cands = list(pathlib.Path("/content/repo").glob("**/models/baseline.py"))
assert cands, "models/baseline.py not found after clone"
ROOT = str(cands[0].parent.parent)
print("ROOT=", ROOT)
os.chdir(ROOT)

In [ ]:
!pip -q install -r requirements.txt
!mkdir -p data
!git clone --depth 1 https://github.com/GabryV00/ThermalUAV2UAV_Dataset.git data/ThermalUAV2UAV_Dataset

In [ ]:
!python models/baseline.py --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --quantize {QUANTIZE} --name {RUN_NAME}

In [ ]:
from pathlib import Path
import shutil

run_reports = Path("reports") / RUN_NAME
compare_dir = Path("reports") / "compare"
print("Report folder:", run_reports if run_reports.exists() else "(missing)")

if SAVE_RUNS_TO_DRIVE:
    dest = Path(DRIVE_ROOT) / RUN_NAME
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(Path("runs") / RUN_NAME, dest)
    drive_reports = Path(DRIVE_ROOT) / "reports" / RUN_NAME
    if drive_reports.exists():
        shutil.rmtree(drive_reports)
    if run_reports.exists():
        shutil.copytree(run_reports, drive_reports)
    if compare_dir.exists():
        drive_cmp = Path(DRIVE_ROOT) / "reports" / "compare"
        drive_cmp.mkdir(parents=True, exist_ok=True)
        for p in compare_dir.glob("*"):
            if p.is_file():
                shutil.copy2(p, drive_cmp / p.name)
    print("Copied run + reports to Drive:", dest)

### After Colab
1. Download `reports/<run>/` and `reports/compare/`.
2. Commit them to the repo.
3. Keep heavy `runs/` and `*.pt` files on Drive (they are gitignored).